# 11 · Parity as ground truth — your backtest is fiction until it matches the broker

> ⏱️ **~30 min** &nbsp;·&nbsp; 🧭 **SOP:** Phase 7 · MQL5 parity (the last gate before live) &nbsp;·&nbsp; 🧩 **Feeds:** every "validated" claim in notebooks 09–10
>
> 🎯 **Goal:** Learn the discipline that actually made KK-MasterVP *reliable* — proving the shipped EA reproduces the C++ backtest **trade-for-trade**, and diffing **config before logic** when it doesn't.
>
> 🔑 **The one thing to remember:** A backtest number is a *claim*, not a fact. It becomes a fact only when an independent implementation (MT5) reproduces it on the same ticks. Until then, every PF you computed in NB 09–10 is the **optimistic** number.

## Notebook 11 in one breath

> **Where we are.** Notebooks 00–10 taught you to *find* an edge: raw ticks → features → discovery →
> costed backtest → walk-forward. But a found edge is a hypothesis about a number. The work that turned
> KK-MasterVP from "the engine liked this config" into "this EA is cleared for forward-test" lives in a
> layer those notebooks never showed you: **parity validation.**

Three ideas, each a hard-won lesson from this project's git history:
1. **The ground-truth ladder** — why Python (and even the C++ engine) is *not* the final tester; MT5 is.
2. **Diff config before logic** — a single wrong input mimics a deep bug. This trap cost *days*. You'll
   reproduce it on the real `.set` files.
3. **Trade-for-trade matching** — what a PASS/FAIL parity gate actually checks, run on a real trade stream.

> 📘 **Why an engineer should care.** This is just **golden-master / reproducibility testing** applied to a
> trading system. You already know it from software: the test that proves two implementations of the same
> spec produce byte-identical output. Here the "two implementations" are the C++ research engine and the
> MQL5 EA, and the "golden master" is the broker's own tick replay.

In [1]:
# --- Standard setup (run me first) -------------------------------------------------
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_root(start: Path) -> Path:
    """Walk up until we see the repo's pipeline/ + data/ — works from any sub-dir."""
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "pipeline").is_dir() and (p / "data").is_dir():
            return p
    raise RuntimeError("repo root (with pipeline/ and data/) not found above " + str(start))

ROOT = _find_root(Path.cwd())
PARITY = ROOT / "research" / "mastervp_parity"   # the real KK-MasterVP parity + WF artifacts
KP     = ROOT / "research" / "kenkem_parity"      # KenKem parity artifacts (config diffs, bars)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 130)
pd.set_option("display.float_format", lambda v: f"{v:,.6g}")
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

print("repo root :", ROOT)
print("parity    :", PARITY)

repo root : /Users/tokyotechies/Workspace/KEM/dquants
parity    : /Users/tokyotechies/Workspace/KEM/dquants/research/mastervp_parity


## Step 1 · The ground-truth ladder — Python is *not* the tester

Recall the four-layer architecture. The crucial point for *trust* is that confidence only ever flows
**downward**, and each layer can lie to the one above it:

```
Layer 1  Python research   vectorbt quick-backtest   ← FAST, but a toy: approximate fills, no real tick path
Layer 2  C++ strategy core pure signal/SL/TP logic   ← deterministic, but still a MODEL of the broker
Layer 3  C++ tick backtester replays real MT5 ticks   ← the TRUE tester... IF its logic == the EA's
Layer 4  MQL5 EA (OnTick)  what actually trades live  ← the ground truth. Nothing below it.
```

> ⚠️ **The mistake the old pipeline made.** "Validated" used to mean *the C++ engine liked a config* — never
> that MT5 reproduced it. That is exactly how an overfit or mis-ported strategy ships: the engine and the EA
> are two different programs, and *nobody checked they agree*. The fix is a gate that forces them to agree on
> the same ticks, the same window, the same `.set`.

The order of trust is strict: **Layer 1 PF < Layer 3 PF < a PASS on the Layer 4 parity gate.** A number from
a higher layer is only a hypothesis about the layer below it.

## Step 2 · Diff config *before* logic — the trap that cost days

When the EA and the engine disagree, every instinct says *"there's a bug in my logic."* Almost always there
isn't — there's a **config mismatch**. One `Inp*` in MT5 differs from the `.set` the engine ran, and a wrong
input is indistinguishable from a deep bug until you diff the inputs.

> 🧨 **The real case (KenKem E1).** Recall was stuck at ~50%. Days went into "the sideways gate over-blocks."
> The actual cause: the engine's `.set` had `E1_MAX_CROSS_AGE=28` but the MT5 run echoed **80** — the *only*
> mismatch out of 193 keys. Fixing the one number took recall **50% → 93.4%**. No logic was ever wrong.

So the **first** thing a parity gate does is diff the two configs. Let's build that diff on the real `.set`
files in the repo.

In [2]:
def load_set(path: Path) -> dict:
    """Parse an MT5 .set / engine --set-all file into {KEY: value} (ignores comments/blanks)."""
    d = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith(";") or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        d[k.strip()] = v.strip()
    return d

def diff_sets(a: Path, b: Path):
    """Return keys whose VALUES differ, plus keys present in only one file."""
    A, B = load_set(a), load_set(b)
    keys = sorted(set(A) | set(B))
    rows = []
    for k in keys:
        va, vb = A.get(k, "<absent>"), B.get(k, "<absent>")
        if va != vb:
            rows.append({"key": k, a.name: va, b.name: vb})
    return pd.DataFrame(rows), len(A), len(B)

# Two real KenKem configs: the engine anchor vs an MT5-echoed gate-trace set.
a = KP / "anchor_E1E2.set"
b = KP / "MT5_E1E2_GATETRACE.set"
if a.exists() and b.exists():
    d, na, nb = diff_sets(a, b)
    print(f"{a.name}: {na} keys   |   {b.name}: {nb} keys   |   {len(d)} differ")
    display(d.head(20))
else:
    print("(sample .set files not present — the technique is the point; see load_set/diff_sets above)")

anchor_E1E2.set: 412 keys   |   MT5_E1E2_GATETRACE.set: 414 keys   |   2 differ


,key,anchor_E1E2.set,MT5_E1E2_GATETRACE.set
0,E1_ARM_TRACE,<absent>,true
1,E1_GATE_TRACE,<absent>,true


> 🧠 **How to read this.** A short diff (one or two keys) is the *good* outcome — it tells you exactly which
> input to reconcile, and you never touch the logic. A *zero*-row diff means config is identical and any
> remaining divergence is genuinely in the code (or the fill model). Either way you've **localized** the
> problem before reading a line of strategy logic. This single habit is the highest-leverage thing in the
> whole parity workflow.

## Step 3 · Worked bug — "the ATR was right, the *smoothing* was wrong"

Config matches but trades still diverge? Now it's the logic — and the bugs are *subtle*. The canonical one:
MT5's built-in `iATR` is a **rolling simple moving average of True Range**, not the textbook **Wilder** ATR
everyone implements. They look identical in a formula sheet and differ by a few percent in practice — enough
to push ~a third of bars into the *wrong* ATR-percentile bucket, which flipped entry gates on and off.

Let's reproduce the divergence on **real XAUUSD M1 bars** so you feel why "close enough" isn't.

> 📘 **Concept — Wilder vs SMA.** True Range `TR = max(high−low, |high−prevClose|, |low−prevClose|)`.
> **Wilder** smooths it recursively: `ATR[i] = ATR[i−1] + (TR[i] − ATR[i−1]) / n` (an EMA with α = 1/n).
> **SMA** smooths it as a flat window: `ATR[i] = mean(TR[i−n+1 .. i])`. Same TR, different memory.

In [3]:
import duckdb
bars_path = KP / "bars_xauusd_M1_kk.csv"
if bars_path.exists():
    # Stream a slice with DuckDB (house style) — enough bars to see the effect, fast to run.
    df = duckdb.sql(f"SELECT open, high, low, close FROM read_csv_auto('{bars_path}') "
                    f"LIMIT 60000").df()
    prev_close = df["close"].shift(1)
    tr = np.maximum(df["high"] - df["low"],
                    np.maximum((df["high"] - prev_close).abs(),
                               (df["low"] - prev_close).abs()))
    n = 14

    # SMA-of-TR (what MT5's iATR actually does)
    atr_sma = tr.rolling(n).mean()

    # Wilder RMA (the textbook ATR almost everyone ports by mistake)
    atr_wilder = tr.copy() * np.nan
    seed = tr.iloc[1:n+1].mean()           # seed at index n with mean of first n TRs
    atr_wilder.iloc[n] = seed
    a = 1.0 / n
    for i in range(n+1, len(tr)):
        atr_wilder.iloc[i] = atr_wilder.iloc[i-1] + a * (tr.iloc[i] - atr_wilder.iloc[i-1])

    cmp = pd.DataFrame({"tr": tr, "atr_sma": atr_sma, "atr_wilder": atr_wilder}).dropna()
    pct_diff = (cmp["atr_wilder"] - cmp["atr_sma"]) / cmp["atr_sma"] * 100
    print(f"bars compared        : {len(cmp):,}")
    print(f"mean |Wilder-SMA| %  : {pct_diff.abs().mean():.2f}%   (looks tiny...)")
    print(f"95th pct |diff| %    : {pct_diff.abs().quantile(0.95):.2f}%")
    print(f"max |diff| %         : {pct_diff.abs().max():.2f}%")
else:
    print("(bars_xauusd_M1_kk.csv not present — formula in the cell is the takeaway)")
    cmp = None

bars compared        : 59,986
mean |Wilder-SMA| %  : 6.88%   (looks tiny...)
95th pct |diff| %    : 17.64%
max |diff| %         : 52.84%


In [4]:
# Why a 'few percent' MATTERS: the strategy gates on the ATR *percentile bucket*, not raw ATR.
# A small value shift near a bucket edge flips the gate -> a different trade decision.
if cmp is not None:
    win = 500   # rolling lookback the gate uses to rank 'is volatility high right now?'
    def rolling_bucket(s, win, q=5):
        # percentile-rank of the latest value within its trailing window, bucketed into q bins
        rank = s.rolling(win).apply(lambda x: (x[:-1] < x[-1]).mean(), raw=True)
        return np.floor(rank * q).clip(0, q-1)
    b_sma    = rolling_bucket(cmp["atr_sma"], win)
    b_wilder = rolling_bucket(cmp["atr_wilder"], win)
    both = pd.DataFrame({"sma": b_sma, "wilder": b_wilder}).dropna()
    disagree = (both["sma"] != both["wilder"]).mean() * 100
    print(f"ATR-percentile bucket DISAGREEMENT (Wilder vs SMA): {disagree:.1f}% of bars")
    print("-> every disagreeing bar is a potential entry-gate flip = a trade the EA takes but the")
    print("   engine doesn't (or vice-versa). 'A few % on ATR' became a double-digit % of WRONG gates.")
    print()
    print("PRODUCTION RESULT after switching the engine to SMA-of-TR (real numbers from this repo):")
    print("  forming-ATR exact match   0.12%  -> 99.93%")
    print("  ATR-percentile exact      31.6%  -> 81.4%")
    print("  entry-gate category agree ~71%   -> 100%")
    print("  E1 matched/missed/overfire 47/31/62 -> 67/11/31")

ATR-percentile bucket DISAGREEMENT (Wilder vs SMA): 23.0% of bars
-> every disagreeing bar is a potential entry-gate flip = a trade the EA takes but the
   engine doesn't (or vice-versa). 'A few % on ATR' became a double-digit % of WRONG gates.

PRODUCTION RESULT after switching the engine to SMA-of-TR (real numbers from this repo):
  forming-ATR exact match   0.12%  -> 99.93%
  ATR-percentile exact      31.6%  -> 81.4%
  entry-gate category agree ~71%   -> 100%
  E1 matched/missed/overfire 47/31/62 -> 67/11/31


> 🧠 **The lesson generalizes.** The bug wasn't in the *signal* — it was in a primitive (ATR) that everyone
> assumes is standard. When porting between two systems, the indicators you trust most are the ones to
> verify first, because a 5% smoothing difference is invisible in isolation and decisive at a gate boundary.
> Other instances of the same class in this repo: a non-series `CopyBuffer` reversing an EMA buffer (off by
> one bar), and a *forming-bar* ATR shrinking the SL by ~7%. All silent; all caught only by trade-level diff.

## Step 4 · The trade-level gate — matching, tolerances, PASS/FAIL

The final gate aligns the engine's trades against MT5's trades on the **same ticks + `.set` + window** and
asks: *do they agree trade by trade?* The matcher is deliberately simple and stdlib-only (it has to run in
CI or a bare shell): greedy nearest-entry-time **within the same direction**, bounded by a small bar-lag
tolerance, after window-filtering the (usually wider) engine run to MT5's entry-time span.

Let's load a **real** locked KK-MasterVP trade stream and look at the columns the gate compares.

In [5]:
tcols = ["entryTimeUTC", "dir", "entry", "riskPrice", "realizedUsd", "exitTag"]
oos = PARITY / "_locked_oos.csv"
if oos.exists():
    tr_df = pd.read_csv(oos, parse_dates=False)
    print(f"locked OOS trade stream: {len(tr_df):,} trades, {tr_df.shape[1]} columns")
    print("\nThe 6 fields the parity gate actually checks per trade:")
    display(tr_df[tcols].head(8))
    print("exit-tag mix (TP / SL-WIN / SL-LOSS / EA):")
    print(tr_df["exitTag"].value_counts().to_string())
else:
    print("(_locked_oos.csv not present)"); tr_df = None

locked OOS trade stream: 892 trades, 21 columns

The 6 fields the parity gate actually checks per trade:


,entryTimeUTC,dir,entry,riskPrice,realizedUsd,exitTag
0,2026.02.01 23:18,S,"4,726.54",31.892,-95.68,SL-LOSS
1,2026.02.02 01:09,S,"4,615.99",38.084,-114.25,SL-LOSS
2,2026.02.02 01:15,S,"4,682.16",41.467,34.63,SL-WIN
3,2026.02.02 01:24,S,"4,689.49",42.503,-85.01,SL-LOSS
4,2026.02.02 03:12,S,"4,692.39",22.103,21.05,SL-WIN
5,2026.02.02 03:15,S,"4,698.77",23.7,-94.8,SL-LOSS
6,2026.02.02 03:30,S,"4,679.56",21.724,20.59,SL-WIN
7,2026.02.02 05:33,S,"4,636.4",20.926,367.13,SL-WIN


exit-tag mix (TP / SL-WIN / SL-LOSS / EA):
exitTag
SL-WIN     464
SL-LOSS    415
TP          13


In [6]:
# A miniature of the real matcher (research/validation/parity_diff.py), to show what 'agree' means.
# We engine-vs-engine match a stream to a +1-bar-lagged copy of itself: a clean PASS by construction,
# which is exactly how the production matcher was smoke-tested before trusting it on MT5 output.
from datetime import timedelta
if tr_df is not None:
    def to_trades(df):
        out = []
        for _, r in df.iterrows():
            out.append((pd.to_datetime(r["entryTimeUTC"], format="%Y.%m.%d %H:%M"),
                        r["dir"], float(r["entry"]), float(r["riskPrice"]),
                        float(r["realizedUsd"]), r["exitTag"]))
        return out
    A = to_trades(tr_df)
    # Simulate the 'other side': same trades, entry stamped 1 bar (5 min) later, tiny price noise.
    B = [(t + timedelta(minutes=5), d, e + 0.001, rk, p, x) for (t, d, e, rk, p, x) in A]
    lag = timedelta(minutes=5)
    usedB = [False] * len(B)
    matched = entry_ok = pnl_ok = 0
    for (t, d, e, rk, p, x) in A:
        best, bestdt = -1, None
        for j, (t2, d2, e2, rk2, p2, x2) in enumerate(B):
            if usedB[j] or d2 != d:
                continue
            dt = abs((t2 - t).total_seconds())
            if dt <= lag.total_seconds() and (bestdt is None or dt < bestdt):
                best, bestdt = j, dt
        if best >= 0:
            usedB[best] = True; matched += 1
            if abs(e - B[best][2]) <= 0.05: entry_ok += 1      # entry tolerance
            if abs(p - B[best][4]) <= 1.0:  pnl_ok += 1        # P&L tolerance ($)
    n = len(A)
    print(f"trades A={n}  matched={matched} ({matched/n:.0%})  entry within tol={entry_ok}  pnl within tol={pnl_ok}")
    verdict = "PASS" if matched == n and entry_ok == n else "FAIL"
    print(f"VERDICT: {verdict}  (engine-vs-engine self-check — proves the matcher chain is sound)")

trades A=892  matched=892 (100%)  entry within tol=892  pnl within tol=892
VERDICT: PASS  (engine-vs-engine self-check — proves the matcher chain is sound)


> 📘 **Concept — what each failure mode means.** The gate's *diagnosis* matters more than its verdict:
>
> | Symptom | Almost always means | Where to look |
> |---|---|---|
> | **Count mismatch** (different # of trades) | a **config** gap | diff the `.set` vs MT5 `inputs_echo.txt` (Step 2) |
> | Matched, but **entry/SL drift** | a logic/indicator port bug | the indicator primitives (Step 3) |
> | Matched, but **P&L / exit-tag drift** | execution-side: fill model, spread, trail granularity | the `mfeR`/`maeR`/`exitTag` columns |
>
> Run order is the whole discipline: **config → entry → exit.** Skipping to "my exit logic is wrong" before
> diffing config is how days get burned.

## 🎯 Recap & your turn

**What you learned**
1. **The ground-truth ladder** — Python is a toy tester, the C++ engine is a model, and only an MT5-parity
   PASS makes a backtest number a *fact*. Confidence flows downward and each layer can lie to the one above.
2. **Diff config before logic** — one wrong input mimics a deep bug; the `.set` diff is the first move, always.
3. **Trade-for-trade matching** — greedy same-direction time-match with explicit entry/P&L tolerances, and a
   failure *taxonomy* (count → config, entry → logic, P&L → execution) that tells you where to look.

> 🔑 Every "PF 1.3" in notebooks 09–10 is a Layer-1 *claim*. This notebook is how the project turns a claim
> into something you'd risk money on.

**🎯 Your turn**
1. **Make the matcher strict.** In Step 4, tighten the entry tolerance to `0.005` and the P&L tolerance to
   `$0.10`. Does the self-check still PASS? Now add bigger noise to `B` — at what point does it FAIL, and
   which counter trips first?
2. **Break config, watch it cascade.** Copy `_locked_oos.csv`, drop every 5th trade, and re-run the matcher.
   You've simulated a count mismatch — confirm the verdict flips and reason about why "count mismatch ⇒ check
   config first" is the right reflex.
3. **Find a real divergence.** Re-run the Step-3 ATR cell with `n=10` and `n=20`. Does the bucket-disagreement
   rate grow or shrink with the lookback? What does that tell you about which ATR period is most parity-fragile?

> ➡️ **Next:** NB 12 — once trades are parity-clean, the remaining way to fool yourself is *overfitting*. We
> recompute walk-forward and Monte-Carlo on the **real** locked KK-MasterVP stream, and confront the drawdown
> number that lied.